<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Rates, Bonds, and Credit

## Chapter 7 · Government Bonds and Sovereign Curve Risk

[Open in Google Colab](https://colab.research.google.com/github/yhilpisch/ratescode/blob/main/notebooks/ch07_government_bonds_sovereign_curve_risk.ipynb)

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Learning goals

Government bonds turn a sovereign curve into prices and curve risk. In this lab you price illustrative benchmark bonds by discounting each annual cash flow at its own interpolated zero rate, read a finite-difference key-rate DV01 from a one-basis-point node bump, and explain which bonds load on which parts of the curve. You keep the pedagogical curve and bond tables separate from the official U.S. Treasury curve-risk panel that anchors the chapter's data discussion, and you state the units, sign, and interpolation choices behind every DV01.

## Prepare the companion repository

This setup cell works with a local checkout or a fresh Google Colab runtime. In Colab it clones the public companion repository, then sets `ROOT` so the chapter code and frozen data snapshots are found without manual path edits.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/yhilpisch/ratescode.git"
candidates = [Path.cwd(), Path.cwd().parent,
              Path("/content/07_ratescode")]
ROOT = next((p.resolve() for p in candidates
             if (p / "code").is_dir()
             and (p / "data").is_dir()), None)
if ROOT is None:
    ROOT = Path("/content/07_ratescode")
    subprocess.run(["git", "clone", "--depth", "1",
                    REPO_URL, str(ROOT)], check=True)

packages = {"numpy": "numpy", "pandas": "pandas",
            "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *missing], check=True)
print(f"Repository: {ROOT}")
print(f"Data files: {len(list((ROOT / 'data').glob('*.csv')))}")


## Run the chapter example

The script prices three illustrative sovereign bonds (2Y, 5Y, and 10Y with 3.5%, 4.0%, and 4.5% annual coupons) off an interpolated zero curve with nodes at 1, 2, 5, and 10 years, then bumps the 5-year node by one basis point and reprices each bond. The printed tables show the curve-implied clean prices per 100 of par and the key-rate DV01 each bond loses under the isolated node shock.

In [ ]:
SCRIPT = ROOT / "code" / "ch07_sovereign_curve_risk.py"
print(SCRIPT.relative_to(ROOT))

result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

## Interpretation checklist

1. Check that each bond prices near par: coupons close to prevailing zero levels give 99.4971, 99.7164, and 101.9796 per 100 of par.
2. Explain why the 2Y note shows a key-rate DV01 of exactly zero under the 5-year bump: its cash-flow dates sit on unbumped nodes, so none of its interpolated discount rates move.
3. Verify why the 5Y note carries the largest key-rate DV01, 0.04383 price points per 100 of par: its principal repayment lands exactly on the bumped node, while the 10Y bond's principal sits on the unbumped 10-year node.
4. Note the conventions behind every number: continuously compounded zero rates, annual cash flows, linear interpolation between nodes, and a positive-DV01 loss convention.
5. Keep the datasets straight: the curve and bond tables are pedagogical snapshots, while the chapter's U.S. Treasury curve-risk panel is an official snapshot with source metadata.

In [ ]:
import pandas as pd

DATA = ROOT / "data"
chapter_files = sorted(DATA.glob("ch07_*.csv"))
summary = pd.DataFrame([
    {"file": path.name,
     "rows": len(frame := pd.read_csv(path)),
     "columns": len(frame.columns),
     "missing_cells": int(frame.isna().sum().sum()),
     "fields": str(list(frame.columns))}
    for path in chapter_files
])
summary


## Mini-lab: Scale DV01 with duration

Under the duration approximation, DV01 is market value times modified duration divided by 10,000, so it scales linearly with both. This lab holds market value fixed and doubles, then doubles again, the modified duration to make the scaling visible, and converts each DV01 into a loss under a 25-basis-point rise. Inputs are illustrative; the official Treasury panel used in the chapter is a separate dataset.

In [ ]:
import pandas as pd

value = 10_000_000  # market value held in each bond (USD)
durations = [2.1, 4.2, 8.4]  # modified durations (years)
bonds = pd.DataFrame({
    "bond": ["2Y note", "5Y note", "10Y bond"],
    "duration": durations,
})
bonds["dv01"] = value * bonds.duration / 10_000  # USD per basis point
shift = 25  # parallel yield rise (basis points)
bonds["loss"] = bonds.dv01 * shift  # USD loss when yields rise
print(bonds.round(2))

### What to observe

The DV01s are 2,100, 4,200, and 8,400 USD per basis point: doubling the duration from 2.1 to 4.2 doubles the DV01, and 8.4 quadruples it. The 25-basis-point rise then costs 52,500, 105,000, and 210,000 USD, exactly 25 times each DV01, which is the linear scaling the duration approximation promises.

## Exercises

1. **Price one bond by hand.** Reproduce the 5Y note price of 99.7164 by discounting its five annual cash flows at the interpolated zero rates. *Hint:* the rates run 3.5%, 3.7%, 3.8%, 3.9%, and 4.0% with continuous compounding.
2. **Bump a different node.** Shock the 10-year node instead of the 5-year and rank the three key-rate DV01s. *Hint:* pass node=10.0 to the script's key-rate function.
3. **Parallel versus key-rate.** Compute the 5Y note's parallel-shift DV01 and compare it with the node-bumped 0.04383 from the script. *Hint:* shift all four curve nodes by one basis point and reprice.
4. **Add a Z-spread.** Reprice the 10Y bond with a 25-basis-point constant spread on every zero rate. *Hint:* multiply each discount factor by exp(-0.0025 * t) and recompute the price.
5. **Inspect the official panel.** Open the U.S. Treasury curve-risk panel and its metadata sidecar and name one field the teaching CSVs lack. *Hint:* official snapshots carry source, retrieval, and schema information.

For more detail, see Chapter 7 of *Python & AI for Rates, Bonds, and Credit*.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">